In [ ]:
# Initialization, styling configuration, and helper utilities
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import torch
from ipywidgets import interact
import ipywidgets as widgets

# Load course helper utilities (local file or direct GitHub fetch for standalone Colab)
try:
    import helpers as hp
except ImportError:
    import urllib.request
    _url = "https://raw.githubusercontent.com/konstantin-schekotihin/dl_course/master/ML-DL/02_deep_learning/helpers.py"
    urllib.request.urlretrieve(_url, "helpers.py")
    import helpers as hp

import importlib
importlib.reload(hp)

hp.setup_theme()

# Gradient Descent & Optimization
## Error Surfaces, Momentum, and Adaptive Learning Rates

This lecture explores gradient-based optimization in deep learning:
- **Error Surface Geometry**: Local quadratic approximations, Hessian matrices, and condition numbers
- **Optimization Obstacles**: Ill-conditioned valleys, ravines, saddle points, and plateaus
- **Descent Paradigms**: Batch Gradient Descent, Mini-batch GD, and Stochastic Gradient Descent (SGD)
- **Accelerated Methods**: Classical Polyak momentum and Nesterov accelerated gradient
- **Adaptive Learning Rates**: AdaGrad, RMSprop, and Adam
- **Practical Techniques**: Learning rate schedules and gradient clipping

# Error Surface Geometry
## The Local Quadratic Approximation

Near any point $\mathbf{w}_0$ in parameter space, the Taylor expansion of the error function $E(\mathbf{w})$ is:
$$E(\mathbf{w}) \approx E(\mathbf{w}_0) + \mathbf{g}^\mathrm{T}(\mathbf{w} - \mathbf{w}_0) + \frac{1}{2}(\mathbf{w} - \mathbf{w}_0)^\mathrm{T} \mathbf{H} (\mathbf{w} - \mathbf{w}_0)$$

where:
- $\mathbf{g} = \nabla E(\mathbf{w}_0)$ is the gradient vector
- $\mathbf{H} = \nabla^2 E(\mathbf{w}_0)$ is the $W \times W$ **Hessian matrix** of second derivatives:
$$H_{ij} = \frac{\partial^2 E}{\partial w_i \partial w_j}$$

At a stationary point $\mathbf{w}^*$ where $\nabla E(\mathbf{w}^*) = \mathbf{0}$:
- **Local Minimum**: All eigenvalues of $\mathbf{H}$ are strictly positive ($\lambda_i > 0$).
- **Local Maximum**: All eigenvalues of $\mathbf{H}$ are strictly negative ($\lambda_i < 0$).
- **Saddle Point**: $\mathbf{H}$ has both positive and negative eigenvalues.

## The Hessian Condition Number

The curvature of the error surface along the eigenvector directions $\mathbf{u}_i$ is governed by the eigenvalues $\lambda_i$:
$$\mathbf{H} \mathbf{u}_i = \lambda_i \mathbf{u}_i$$

The **condition number** $\kappa$ of the Hessian matrix is the ratio of extreme eigenvalues:
$$\kappa = \frac{\lambda_\mathrm{max}}{\lambda_\mathrm{min}}$$

- **Well-Conditioned Surfaces ($\kappa \approx 1$)**:
  Error contours are nearly spherical. Gradient descent steps point directly toward the minimum.
- **Ill-Conditioned Ravines ($\kappa \gg 1$)**:
  Error contours form elongated elliptical valleys. The gradient is nearly orthogonal to the direction of the minimum, causing standard gradient descent to oscillate wildly across the steep walls while making negligible progress along the base.

## High-Dimensional Optimization Landscapes

In high-dimensional parameter spaces ($W \sim 10^6 - 10^9$), intuition from 2D and 3D surfaces fails:

1. **Proliferation of Saddle Points**:
   For a stationary point to be a local minimum, all $W$ eigenvalues must be positive. If signs are random, the probability is $2^{-W} \approx 0$. High-dimensional critical points are almost exclusively **saddle points**.
2. **Escaping Plateaus**:
   Near saddle points, gradients vanish along multiple directions, causing optimization to stall.
3. **Stochastic Perturbation**:
   Mini-batch noise in Stochastic Gradient Descent (SGD) provides the thermal fluctuation necessary to escape saddle points and shallow local traps.

# Gradient Descent Paradigms
## Batch vs Stochastic vs Mini-Batch

Gradient descent updates parameters iteratively in the negative gradient direction:
$$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} - \eta \mathbf{g}_\tau$$

| Paradigm | Gradient Estimator $\mathbf{g}_\tau$ | Variance | Computational Cost per Step |
| :--- | :--- | :---: | :---: |
| **Batch GD** | $\frac{1}{N} \sum_{n=1}^N \nabla E_n(\mathbf{w})$ | Zero | $\mathcal{O}(N)$ (Intractable for large $N$) |
| **SGD** | $\nabla E_n(\mathbf{w})$ (single sample) | High | $\mathcal{O}(1)$ (Fast, high gradient noise) |
| **Mini-Batch GD** | $\frac{1}{B} \sum_{b=1}^B \nabla E_{n_b}(\mathbf{w})$ | Moderate | $\mathcal{O}(B)$ (GPU vectorization + stable gradients) |

Mini-batch GD with batch size $B \in \{32, \dots, 512\}$ represents the universal workhorse of deep learning.

# Momentum Methods
## Dampening Oscillations with the Heavy-Ball Method

To suppress transverse oscillations in ill-conditioned valleys, Boris Polyak (1964) introduced the **momentum method** (analogous to a heavy ball rolling down a potential surface with friction):

$$\mathbf{v}^{(\tau+1)} = \beta \mathbf{v}^{(\tau)} - \eta \nabla E(\mathbf{w}^{(\tau)})$$
$$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} + \mathbf{v}^{(\tau+1)}$$

where:
- $\mathbf{v}$ is the velocity vector (exponentially decaying moving average of past gradients)
- $\beta \in [0, 1)$ is the momentum decay coefficient (typically $\beta = 0.9$)
- $\eta > 0$ is the learning rate

<small>📖 *Source: [Polyak, B. T. (1964). Some methods of speeding up the convergence of iteration methods. USSR Computational Mathematics and Mathematical Physics, 4(5), 1–17](https://doi.org/10.1016/0041-5553(64)90137-5)*</small>

> 💡 **Geometric Effect:** Successive oscillatory gradient components cancel each other out, while consistent downhill components compound, accelerating convergence along the valley base by an effective factor of $\frac{1}{1 - \beta}$.

## Nesterov Accelerated Gradient (NAG)

Yurii Nesterov (1983) refined classical momentum by evaluating the gradient not at the current position $\mathbf{w}^{(\tau)}$, but at the **look-ahead position** $\mathbf{w}^{(\tau)} + \beta \mathbf{v}^{(\tau)}$:

$$\mathbf{v}^{(\tau+1)} = \beta \mathbf{v}^{(\tau)} - \eta \nabla E\left(\mathbf{w}^{(\tau)} + \beta \mathbf{v}^{(\tau)}\right)$$
$$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} + \mathbf{v}^{(\tau+1)}$$

```
Polyak Momentum:     w ---> [ + beta * v ] ---> [ - eta * grad(w) ] = w_new
Nesterov Momentum:   w ---> [ Look-ahead: w + beta * v ] ---> [ - eta * grad(look-ahead) ] = w_new
```

> 📘 **Corrective Anticipation:** NAG computes the gradient where the momentum step is about to take the parameters, allowing it to brake before overshooting sharp turns.

# Adaptive Learning Rates
## Scaling Gradients per Parameter

Different parameters often experience wildly different gradient scales (e.g. frequent vs infrequent features).

1. **AdaGrad (Duchi et al. 2011)**:
   Accumulates squared gradients: $s_j^{(\tau)} = s_j^{(\tau-1)} + g_j^2$.
   $$w_j^{(\tau+1)} = w_j^{(\tau)} - \frac{\eta}{\sqrt{s_j^{(\tau)}} + \epsilon} g_j$$
   *Defect*: $s_j$ accumulates monotonically, causing the effective step size to shrink prematurely to zero.

2. **RMSprop (Tieleman & Hinton 2012)**:
   Replaces unbounded accumulation with an Exponential Moving Average (EMA):
   $$s_j^{(\tau+1)} = \beta_2 s_j^{(\tau)} + (1 - \beta_2) \left(g_j^{(\tau)}\right)^2$$
   $$w_j^{(\tau+1)} = w_j^{(\tau)} - \frac{\eta}{\sqrt{s_j^{(\tau+1)}} + \epsilon} g_j^{(\tau)}$$
   Scaling by $\frac{1}{\sqrt{s_j}}$ normalizes updates, ensuring equal progression along steep and flat coordinates.

## The Adam Optimizer
## Adaptive Moment Estimation

Kingma and Ba (2014) combined **first-moment tracking (momentum)** with **second-moment tracking (RMSprop)**, adding explicit **bias correction**:

1. Update biased first and second moment estimates:
   $$\mathbf{m}^{(\tau)} = \beta_1 \mathbf{m}^{(\tau-1)} + (1 - \beta_1) \mathbf{g}_\tau$$
   $$\mathbf{v}^{(\tau)} = \beta_2 \mathbf{v}^{(\tau-1)} + (1 - \beta_2) \mathbf{g}_\tau^2$$

2. Correct initialization bias toward zero (crucial for early iterations):
   $$\hat{\mathbf{m}}^{(\tau)} = \frac{\mathbf{m}^{(\tau)}}{1 - \beta_1^\tau}, \quad \hat{\mathbf{v}}^{(\tau)} = \frac{\mathbf{v}^{(\tau)}}{1 - \beta_2^\tau}$$

3. Update parameters:
   $$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} - \frac{\eta}{\sqrt{\hat{\mathbf{v}}^{(\tau)}} + \epsilon} \hat{\mathbf{m}}^{(\tau)}$$

Standard defaults: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

<small>📖 *Source: [Bishop, C. M., & Bishop, H. (2024). Deep Learning: Foundations and Concepts. Springer (Section 7.4.3)](https://doi.org/10.1007/978-3-031-45468-4)*</small>

## Practical Techniques: Schedules & Clipping

1. **Learning Rate Schedules**:
   Maintaining a fixed $\eta$ throughout training is suboptimal: initial exploration requires large steps, whereas late fine-tuning requires small steps.
   - **Step Decay**: Multiply $\eta$ by $\gamma \approx 0.1$ every $k$ epochs.
   - **Cosine Annealing (Loshchilov & Hutter 2016)**:
     $$\eta_t = \eta_\mathrm{min} + \frac{1}{2}(\eta_\mathrm{max} - \eta_\mathrm{min})\left(1 + \cos\left(\frac{t}{T}\pi\right)\right)$$
   - **Linear Warmup**: Start from small $\eta$ for the first few epochs to stabilize early moments.

2. **Gradient Clipping**:
   In deep recurrent and transformer networks, gradients can explode ($\|\mathbf{g}\| \to \infty$).
   We clip the global norm:
   $$\mathbf{g} \leftarrow \mathbf{g} \cdot \min\left(1, \frac{\theta}{\|\mathbf{g}\|}\right)$$

In [ ]:
# Simulation of optimization trajectories on an ill-conditioned quadratic valley
def simulate_optimizers(opt_type='sgd', lr=0.08, momentum=0.85):
    # Loss: f(x1, x2) = 0.1 * x1^2 + 2.0 * x2^2 (Condition number kappa = 20)
    w = torch.tensor([-3.5, 2.5], requires_grad=True)
    
    if opt_type == 'sgd':
        opt = torch.optim.SGD([w], lr=lr)
    elif opt_type == 'momentum':
        opt = torch.optim.SGD([w], lr=lr, momentum=momentum)
    elif opt_type == 'rmsprop':
        opt = torch.optim.RMSprop([w], lr=lr)
    elif opt_type == 'adam':
        opt = torch.optim.Adam([w], lr=lr)
        
    trajectory = [w.detach().numpy().copy()]
    for _ in range(50):
        opt.zero_grad()
        loss = 0.1 * w[0]**2 + 2.0 * w[1]**2
        loss.backward()
        opt.step()
        trajectory.append(w.detach().numpy().copy())
        
    traj = np.array(trajectory)
    
    x1 = np.linspace(-4, 4, 150)
    x2 = np.linspace(-3, 3, 150)
    X1, X2 = np.meshgrid(x1, x2)
    Z = 0.1 * X1**2 + 2.0 * X2**2
    
    plt.figure(figsize=(7, 4.2))
    plt.contour(X1, X2, Z, levels=np.logspace(-1, 2, 15), cmap='viridis', alpha=0.5)
    plt.plot(traj[:, 0], traj[:, 1], 'ro-', lw=1.8, ms=4, label=f'{opt_type.upper()} Path')
    plt.plot(0, 0, 'r*', ms=12, label='Minimum (0, 0)')
    plt.title(f"Optimization Trajectory on Ill-Conditioned Ravine (kappa=20): {opt_type.upper()}", fontsize=11)
    plt.xlabel("x_1 (Slow coordinate)")
    plt.ylabel("x_2 (Fast coordinate)")
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive comparison of SGD, Momentum, RMSprop, and Adam on ill-conditioned ravine
interact(simulate_optimizers, opt_type=['sgd', 'momentum', 'rmsprop', 'adam'], 
         lr=widgets.FloatSlider(min=0.01, max=0.25, step=0.01, value=0.08),
         momentum=widgets.FloatSlider(min=0.5, max=0.95, step=0.05, value=0.85));

# Summary: Gradient Descent & Optimization (Part 1)

In this lecture, we analyzed the mechanics of gradient-based optimization:

1. **Error Surface Geometry**:
   - The Hessian condition number $\kappa = \lambda_\mathrm{max}/\lambda_\mathrm{min}$ governs the eccentricity of error contours.
   - Ill-conditioned valleys cause standard gradient descent to oscillate transversely while making slow progress along the valley floor.

2. **Momentum Acceleration**:
   - Polyak momentum accumulates velocity $\mathbf{v}^{(\tau+1)} = \beta \mathbf{v}^{(\tau)} - \eta \mathbf{g}$, averaging out oscillations and accelerating along consistent gradients.
   - Nesterov acceleration incorporates look-ahead gradient corrections.

## Summary: Gradient Descent & Optimization (Part 2)

3. **Adaptive Scaling (RMSprop & Adam)**:
   - RMSprop tracks exponential moving averages of squared gradients to adapt step sizes per coordinate.
   - Adam integrates first-moment momentum with second-moment scaling, applying explicit bias corrections for robust early-phase optimization.

4. **Practical Refinements**:
   - Cosine annealing and warmup schedules systematically tune step sizes across training stages.
   - Gradient clipping prevents catastrophic exploding gradient steps in deep architectures.